# 05 - Weekly Performance Trends

This notebook generates a line graph showing rolling 7-day on-time performance.

**Features:**
- Rolling average line with trend visualization
- Daily data points with scatter markers
- Comparison bands (target vs actual)
- Interactive hover with full statistics

**Output:** `static/plots/weekly_trends.html`

In [6]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
OUTPUT_DIR = os.path.join(project_root, 'static', 'plots')

print(f"Database exists: {os.path.exists(DB_PATH)}")

Database exists: True


## Configuration

In [7]:
# ====================
# CONFIGURATION
# ====================

# Number of days to show (None = all)
DAYS_TO_SHOW = 90

# Rolling average window
ROLLING_WINDOW = 7

# Target on-time percentage (for reference line)
TARGET_ON_TIME = 90.0

# Colors
COLORS = {
    'daily': '#58a6ff',       # Light blue for daily points
    'rolling': '#3fb950',     # Green for rolling average
    'target': '#f85149',      # Red for target line
    'fill': 'rgba(63, 185, 80, 0.1)',
}

THEME = {
    'bg_color': '#0d1117',
    'text_color': '#c9d1d9',
    'grid_color': '#21262d',
}

FIGURE_WIDTH = 1000
FIGURE_HEIGHT = 450

## Load and Process Data

In [8]:
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_arrivals():
    """Load and process arrival data."""
    conn = sqlite3.connect(DB_PATH)
    
    date_filter = ""
    if DAYS_TO_SHOW:
        cutoff = (datetime.now() - timedelta(days=DAYS_TO_SHOW)).strftime('%Y-%m-%d')
        date_filter = f" WHERE timestamp >= '{cutoff}'"
    
    query = f"""
    SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp
    FROM train_locations
    {date_filter}
    """
    raw_df = pd.read_sql_query(query, conn)
    conn.close()
    
    if raw_df.empty:
        return pd.DataFrame()
    
    raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
    raw_df = raw_df.dropna(subset=['timestamp'])
    print(f"Loaded {len(raw_df):,} records")
    
    # Load GTFS
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    
    # Type conversion
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    
    # Merge
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], 
                  on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_lat', 'stop_lon']], 
                  on='stop_id', how='inner')
    
    df['distance'] = df.apply(lambda r: haversine(
        r['vehicle_lat'], r['vehicle_lon'], r['stop_lat'], r['stop_lon']
    ), axis=1)
    
    df['date'] = df['timestamp'].dt.date
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    # Arrivals
    min_dist = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_dist, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    
    arrivals['delay_min'] = arrivals.apply(
        lambda r: calculate_time_difference(r['arrival_time'], r['timestamp']), axis=1
    )
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    arrivals['on_time'] = arrivals['delay_min'] <= 4
    
    print(f"Processed {len(arrivals):,} arrivals")
    return arrivals

arrivals = load_arrivals()

Loaded 952,442 records
Processed 151,684 arrivals


## Calculate Daily and Rolling Statistics

In [9]:
def calculate_trends(arrivals):
    """Calculate daily and rolling average statistics."""
    if arrivals.empty:
        return pd.DataFrame()
    
    # Daily stats
    daily = arrivals.groupby('date').agg(
        on_time_count=('on_time', 'sum'),
        total_count=('on_time', 'count'),
        avg_delay=('delay_min', 'mean')
    ).reset_index()
    
    daily['on_time_pct'] = (daily['on_time_count'] / daily['total_count'] * 100).round(1)
    daily['date'] = pd.to_datetime(daily['date'])
    daily = daily.sort_values('date')
    
    # Rolling average
    daily['rolling_avg'] = daily['on_time_pct'].rolling(
        window=ROLLING_WINDOW, min_periods=1
    ).mean().round(1)
    
    # Week-over-week change
    daily['wow_change'] = daily['rolling_avg'].diff(periods=7).round(1)
    
    print(f"Stats for {len(daily)} days")
    print(f"Current 7-day avg: {daily['rolling_avg'].iloc[-1]:.1f}%")
    
    return daily

trends = calculate_trends(arrivals)
trends.tail(10)

Stats for 89 days
Current 7-day avg: 91.3%


,date,on_time_count,total_count,avg_delay,on_time_pct,rolling_avg,wow_change
79,2025-12-19,1628,1886,2.700256,86.3,92.7,4.0
80,2025-12-20,746,824,-0.625627,90.5,93.0,4.3
81,2025-12-21,1330,1379,-0.932487,96.4,93.3,2.8
82,2025-12-22,1807,1926,-1.346080,93.8,93.1,2.5
83,2025-12-23,1832,1931,-1.603806,94.9,93.0,1.6
84,2025-12-24,1369,1524,-0.623808,89.8,91.9,-0.4
85,2025-12-25,1185,1383,-0.047457,85.7,91.1,-1.2
86,2025-12-26,1799,1939,-1.333763,92.8,92.0,-0.7
87,2025-12-27,1371,1404,-1.347626,97.6,93.0,0.0
88,2025-12-28,231,273,0.107448,84.6,91.3,-2.0


## Create Line Chart

In [10]:
def create_trend_chart(trends):
    """Create trend line chart."""
    fig = go.Figure()
    
    # Daily scatter points (smaller, semi-transparent)
    fig.add_trace(go.Scatter(
        x=trends['date'],
        y=trends['on_time_pct'],
        mode='markers',
        name='Daily',
        marker=dict(
            color=COLORS['daily'],
            size=6,
            opacity=0.5
        ),
        hovertemplate=(
            '<b>%{x|%b %d, %Y}</b><br>'
            'On-Time: <b>%{y:.1f}%</b><br>'
            '<extra></extra>'
        )
    ))
    
    # Rolling average line
    fig.add_trace(go.Scatter(
        x=trends['date'],
        y=trends['rolling_avg'],
        mode='lines',
        name=f'{ROLLING_WINDOW}-Day Avg',
        line=dict(color=COLORS['rolling'], width=3),
        fill='tozeroy',
        fillcolor=COLORS['fill'],
        hovertemplate=(
            '<b>%{x|%b %d, %Y}</b><br>'
            f'{ROLLING_WINDOW}-Day Avg: <b>%{{y:.1f}}%</b><br>'
            '<extra></extra>'
        )
    ))
    
    # Target reference line
    fig.add_hline(
        y=TARGET_ON_TIME,
        line_dash='dash',
        line_color=COLORS['target'],
        annotation_text=f'Target ({TARGET_ON_TIME}%)',
        annotation_position='right',
        annotation_font=dict(color=COLORS['target'])
    )
    
    # Calculate trend summary
    current_avg = trends['rolling_avg'].iloc[-1]
    prev_avg = trends['rolling_avg'].iloc[-8] if len(trends) > 7 else trends['rolling_avg'].iloc[0]
    change = current_avg - prev_avg
    trend_emoji = '📈' if change > 0 else '📉' if change < 0 else '➡️'
    trend_text = f"{trend_emoji} {'+' if change > 0 else ''}{change:.1f}% vs last week"
    
    # Date range
    date_range = f"{trends['date'].min().strftime('%b %d')} - {trends['date'].max().strftime('%b %d, %Y')}"
    
    fig.update_layout(
        title=dict(
            text=f'📈 Weekly Performance Trend<br><span style="font-size:14px;color:{THEME["text_color"]}">{date_range} | Current: {current_avg:.1f}% | {trend_text}</span>',
            font=dict(size=22, color=THEME['text_color']),
            x=0.5,
            xanchor='center'
        ),
        xaxis=dict(
            title='Date',
            tickfont=dict(color=THEME['text_color']),
            # titlefont=dict(color=THEME['text_color']),
            gridcolor=THEME['grid_color'],
            tickformat='%b %d',
        ),
        yaxis=dict(
            title='On-Time Percentage',
            ticksuffix='%',
            tickfont=dict(color=THEME['text_color']),
            # titlefont=dict(color=THEME['text_color']),
            gridcolor=THEME['grid_color'],
            range=[50, 100],  # Focus on relevant range
        ),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=-0.2,
            xanchor='center',
            x=0.5,
            font=dict(color=THEME['text_color']),
        ),
        plot_bgcolor=THEME['bg_color'],
        paper_bgcolor=THEME['bg_color'],
        height=FIGURE_HEIGHT,
        width=FIGURE_WIDTH,
        margin=dict(l=60, r=40, t=100, b=80),
        hovermode='x unified'
    )
    
    return fig

fig = create_trend_chart(trends)
fig.show()

## Trend Analysis

In [11]:
def analyze_trends(trends):
    """Analyze performance trends."""
    print("=" * 50)
    print("📈 TREND ANALYSIS")
    print("=" * 50)
    
    # Overall stats
    print(f"\n📊 Overall Average: {trends['on_time_pct'].mean():.1f}%")
    print(f"📊 Current 7-Day Avg: {trends['rolling_avg'].iloc[-1]:.1f}%")
    
    # Best/worst periods
    best_day = trends.loc[trends['on_time_pct'].idxmax()]
    worst_day = trends.loc[trends['on_time_pct'].idxmin()]
    
    print(f"\n✅ Best Day: {best_day['date'].strftime('%b %d')} ({best_day['on_time_pct']:.1f}%)")
    print(f"❌ Worst Day: {worst_day['date'].strftime('%b %d')} ({worst_day['on_time_pct']:.1f}%)")
    
    # Recent trend
    if len(trends) >= 14:
        recent = trends.tail(7)['rolling_avg'].mean()
        previous = trends.iloc[-14:-7]['rolling_avg'].mean()
        change = recent - previous
        direction = "improving" if change > 0 else "declining" if change < 0 else "stable"
        print(f"\n📌 Recent Trend: {direction} ({'+' if change > 0 else ''}{change:.1f}%)")
    
    # Days meeting target
    above_target = (trends['on_time_pct'] >= TARGET_ON_TIME).sum()
    pct_above = above_target / len(trends) * 100
    print(f"\n🎯 Days at {TARGET_ON_TIME}% target: {above_target}/{len(trends)} ({pct_above:.1f}%)")

analyze_trends(trends)

📈 TREND ANALYSIS

📊 Overall Average: 91.6%
📊 Current 7-Day Avg: 91.3%

✅ Best Day: Nov 27 (99.0%)
❌ Worst Day: Oct 04 (63.7%)

📌 Recent Trend: declining (-0.0%)

🎯 Days at 90.0% target: 66/89 (74.2%)


## Export

In [12]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'weekly_trends.html')
fig.write_html(output_path, include_plotlyjs='cdn')
print(f"✅ Exported to: {output_path}")

✅ Exported to: d:\caltrain-tracker\static\plots\weekly_trends.html
